# Dehazing Evaluation
Scores every method in `outputs/` against GT with `dehazing/evaluation`.
Run the method notebooks in `dehazing/methods/` first: each one writes its dehazed images to `MyDrive/outputs/<method>/`.
Use a GPU runtime (LPIPS, NIQE and BRISQUE run faster).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyiqa
!rm -rf /content/vx
!git clone -q --depth 1 https://github.com/fahimrayhan/vidhance-extendview.git /content/vx

import sys
sys.path.insert(0, "/content/vx/dehazing")
from evaluation import evaluate

## Paths
Same paths as the method notebooks.

In [ ]:
from pathlib import Path

DRIVE     = "/content/drive/MyDrive"
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # hazy/ + GT/, created by the method notebooks
OUTPUTS   = Path(f"{DRIVE}/outputs")   # one folder of dehazed images per method

## Methods to evaluate
Every folder in `outputs/` is one method. Remove a name from the dict to skip it.

In [ ]:
methods = {p.name: str(p) for p in sorted(OUTPUTS.iterdir()) if p.is_dir()}
print(len(methods), "methods:", list(methods))

## Evaluate

In [ ]:
per_image, summary = evaluate(DATA_ROOT, methods)
summary

In [ ]:
per_image

## Save
The file names include the number of methods, so they always match their content.

In [ ]:
n = len(methods)
summary.to_csv(OUTPUTS / f"summary_{n}methods.csv")
per_image.to_csv(OUTPUTS / f"per_image_{n}methods.csv", index=False)
print("saved:", f"summary_{n}methods.csv", f"per_image_{n}methods.csv")

## How to read the results
- **Ranking**: PSNR ↑, SSIM ↑, CIEDE2000 ↓, LPIPS ↓ (compared with GT)
- **Diagnosis**: e ↑ and r ↑ (edges and contrast gained), sigma ↓ (pixels pushed to pure black/white; above 0.001 means over-processing)
- **No-reference** (NIQE ↓, BRISQUE ↓): unreliable on haze, use only when there is no GT
- The `hazy` row is the hazy input scored against GT, i.e. the score of doing nothing